In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Selected device: {device}")

if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"Current GPU index: {torch.cuda.current_device()}")
    print(f"Current GPU name: {torch.cuda.get_device_name(torch.cuda.current_device())}")

    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        total_vram_gb = props.total_memory / (1024 ** 3)
        print(f"GPU {i} total VRAM: {total_vram_gb:.2f} GB")

    current = torch.cuda.current_device()
    allocated_gb = torch.cuda.memory_allocated(current) / (1024 ** 3)
    reserved_gb = torch.cuda.memory_reserved(current) / (1024 ** 3)
    print(f"GPU {current} allocated VRAM: {allocated_gb:.2f} GB")
    print(f"GPU {current} reserved VRAM: {reserved_gb:.2f} GB")

CUDA available: True
Selected device: cuda
GPU count: 1
Current GPU index: 0
Current GPU name: NVIDIA GeForce GTX 1650 Ti
GPU 0 total VRAM: 3.63 GB
GPU 0 allocated VRAM: 0.00 GB
GPU 0 reserved VRAM: 0.00 GB


In [2]:
import csv

def load_csv_data(filepath):
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            data.append(row)
    return data

def format_input(entry):
    return (
        f"### Instruction:\nConvert the following text to brainrot slang.\n\n"
        f"### Input:\n{entry['original_message']}"
    )

In [3]:
import torch

class InstructionDataset(torch.utils.data.Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['brainrot_message']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [4]:
import tiktoken
from torch.utils.data import random_split

tokenizer = tiktoken.get_encoding("gpt2")
data = load_csv_data("../data/synthetic/data.csv")

dataset = InstructionDataset(data, tokenizer)

train_size = int(0.85 * len(dataset))
val_size = int(0.05 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_data, val_data, test_data = random_split(dataset, [train_size, val_size, test_size])

print(f"Total: {len(dataset)}, Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

Total: 4080, Train: 3468, Val: 204, Test: 408


In [5]:
from torch.nn.utils.rnn import pad_sequence

def custom_collate(batch, pad_token_id=50256, device="cpu"):
    batch_tensors = [torch.tensor(item) for item in batch]
    padded = pad_sequence(batch_tensors, batch_first=True, padding_value=pad_token_id)

    # Build a mask: True for real tokens + one trailing pad (the EOS target)
    lengths = torch.tensor([len(item) for item in batch])

    # positions beyond length+1 are pure padding to ignore

    positions = torch.arange(padded.size(1)).unsqueeze(0)  # (1, seq_len)
    
    target_mask = positions >= lengths.unsqueeze(1)  # True for padding positions

    targets_tensor = torch.roll(padded, -1, dims=1).clone()

    # Set last real token's target to pad_token_id (EOS)
    targets_tensor[torch.arange(len(batch)), lengths - 1] = pad_token_id

    # Mask only pure padding positions
    targets_tensor[target_mask] = -100

    return padded.to(device), targets_tensor.to(device)

In [6]:
batch = (
    [1,2,3,4,5,6],
    [7,8,9],
    [10,11]
)
inputs, targets = custom_collate(batch)
print(inputs)
print(targets)

tensor([[    1,     2,     3,     4,     5,     6],
        [    7,     8,     9, 50256, 50256, 50256],
        [   10,    11, 50256, 50256, 50256, 50256]])
tensor([[    2,     3,     4,     5,     6, 50256],
        [    8,     9, 50256,  -100,  -100,  -100],
        [   11, 50256,  -100,  -100,  -100,  -100]])


In [7]:
inputs, targets = custom_collate(dataset, device=device)
print(inputs)
print(targets)

tensor([[21017, 46486,    25,  ..., 50256, 50256, 50256],
        [21017, 46486,    25,  ..., 50256, 50256, 50256],
        [21017, 46486,    25,  ..., 50256, 50256, 50256],
        ...,
        [21017, 46486,    25,  ..., 50256, 50256, 50256],
        [21017, 46486,    25,  ..., 50256, 50256, 50256],
        [21017, 46486,    25,  ..., 50256, 50256, 50256]], device='cuda:0')
tensor([[46486,    25,   198,  ...,  -100,  -100,  -100],
        [46486,    25,   198,  ...,  -100,  -100,  -100],
        [46486,    25,   198,  ...,  -100,  -100,  -100],
        ...,
        [46486,    25,   198,  ...,  -100,  -100,  -100],
        [46486,    25,   198,  ...,  -100,  -100,  -100],
        [46486,    25,   198,  ...,  -100,  -100,  -100]], device='cuda:0')


In [8]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_loader = DataLoader(
    train_data,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=True,
    drop_last=True,
    collate_fn=custom_collate
)
val_loader = DataLoader(
    val_data,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    drop_last=False,
    collate_fn=custom_collate
)
test_loader = DataLoader(
    test_data,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    drop_last=False,
    collate_fn=custom_collate
)

In [9]:
for batch in train_loader:
    inputs, targets = batch
    print(f"Batch input shape: {inputs.shape}, Batch target shape: {targets.shape}")
    break

Batch input shape: torch.Size([8, 196]), Batch target shape: torch.Size([8, 196])


In [10]:
import sys
import os
sys.path.append(os.path.abspath(".."))
from download_model import download_and_load_gpt2
from architecture import GPTModel, load_weights_into_gpt

BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-small (124M)"

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval();

I0000 00:00:1782049773.347479   23102 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782049775.198356   23102 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


File already exists and is up-to-date: gpt2/124M/checkpoint
File already exists and is up-to-date: gpt2/124M/encoder.json
File already exists and is up-to-date: gpt2/124M/hparams.json
File already exists and is up-to-date: gpt2/124M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/124M/model.ckpt.index
File already exists and is up-to-date: gpt2/124M/model.ckpt.meta
File already exists and is up-to-date: gpt2/124M/vocab.bpe


W0000 00:00:1782049786.812167   23102 cpu_allocator_impl.cc:82] Allocation of 154389504 exceeds 10% of free system memory.


In [11]:
from architecture import (
    calc_loss_loader,
    train_model_simple
)

model.to(device)
torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader,model,device, num_batches=5)
    val_loss = calc_loss_loader(val_loader,model,device, num_batches=5)

print('Training loss', train_loss)
print('Validation loss' , val_loss)


Training loss 4.71547269821167
Validation loss 4.796112823486328


In [12]:
import time

optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)

num_epochs = 2

val_entry = val_data.dataset.data[val_data.indices[0]]
start_context = format_input(val_entry)

start_time = time.time()
train_losses, val_losses, tokens_seen = train_model_simple(
    model=model, train_loader=train_loader, val_loader=val_loader,
    optimizer=optimizer, device=device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=start_context, tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f'Executed in {execution_time_minutes:.2f} min')


OutOfMemoryError: CUDA out of memory. Tried to allocate 298.00 MiB. GPU 0 has a total capacity of 3.63 GiB of which 236.94 MiB is free. Including non-PyTorch memory, this process has 3.39 GiB memory in use. Of the allocated memory 3.15 GiB is allocated by PyTorch, and 172.03 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)